# 12 - Article figures and all tables in one Word file

Reads only files that notebooks 01–11 already wrote; nothing is recomputed, the analysis plan is not touched and the
check of notebook 08 is unaffected. Everything for the manuscript goes to outputs/article/, numbered in the order
of first citation in the manuscript:

| File | Content |
|---|---|
| Fig1_study_design | design of the study (schematic; also .svg with editable text) |
| Fig2_participant_flow | participant flow |
| Fig3_roc_curves | ROC curves |
| Fig4_RQ1_incremental_value | odds ratios + null-bootstrap distributions |
| Fig5_RQ2_confounding_by_design | confounding-free AUCs + covariate balance (Love plots) |
| Fig6_RQ3_transportability | calibration of the frozen models and post hoc recalibration |
| Fig7_feature_consistency | each feature beyond the demographics in both cohorts |
| FigS1_montage_error | angular error of the BrainLat 10-20 correspondence (notebook 11) |
| FigS2_group_spectra | group-average spectra (notebook 11) |
| FigS3_e_value, FigS4_domain_shift | E-value; domain shift |
| FigS5_hyperparameters | hyperparameter sensitivity (notebook 10) |
| article_tables.docx | Tables 1–5 and S1–S12, the TRIPOD+AI checklist and draft figure legends |
| supplementary_files/ | the analysis plan and the two post hoc amendments (JSON) |

article_tables.docx is written again every time this notebook runs: copy it elsewhere before editing it by hand.
Supplementary items whose notebook has not run (10 or 11) are skipped with a warning.

All figures use the sans-serif lettering recommended by Springer; PDF (vector, fonts embedded) and PNG (300 dpi).

**Time:** under a minute.

In [ ]:
# Setup: make the eegad package importable (also when it was not installed with `pip install -e .`)
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=RuntimeWarning)   # MNE notices about head size and 'boundary' events
warnings.filterwarnings("ignore", category=FutureWarning)

_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "analysis_plan.json").exists()), None)
if _root is None:
    raise FileNotFoundError("Open this notebook from inside the repository (eeg-ad-demographics/notebooks).")
if str(_root / "src") not in sys.path:
    sys.path.insert(0, str(_root / "src"))

In [ ]:
import re
import shutil

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from eegad.article_figures import (article_style, domain_shift_rows, feature_rows, fig_domain_shift, fig_evalue,
                                   fig_features, fig_flow, fig_hyperparameters, fig_montage_error, fig_rq1, fig_rq2,
                                   fig_rq3, fig_spectra)
from eegad.article_tables import write_article_tables
from eegad.config import load_context
from eegad.data import load_table
from eegad.design_figure import study_design_figure
from eegad.figures import roc_figure
from eegad.reporting import read_json
from IPython.display import display

ctx = load_context("12")
out, plan = ctx.out, ctx.plan
ART = out / "article"
ART.mkdir(exist_ok=True)
need = [out / "results" / "incremental.json", out / "results" / "transport.json",
        out / "tables" / "T2b_confounding_by_design.csv", out / "post_hoc" / "recalibration.json"]
missing = [str(f) for f in need if not f.exists()]
if missing:
    raise FileNotFoundError(f"Run notebooks 01-09 first; missing: {missing}")
inc = read_json(out / "results" / "incremental.json")
tr = read_json(out / "results" / "transport.json")
keys = read_json(out / "results" / "key_results.json")
rec = read_json(out / "post_hoc" / "recalibration.json")
if rec["mode"] != ctx.tag:
    raise RuntimeError(f"Notebook 09 ran in {rec['mode']} mode and this run is {ctx.tag}: run notebook 09 again.")


def save(fig, name):
    fig.savefig(ART / f"{name}.pdf")
    fig.savefig(ART / f"{name}.png", dpi=300)
    if b"/Type3" in (ART / f"{name}.pdf").read_bytes():
        print(f"WARNING: {name}.pdf contains Type 3 fonts")
    plt.show()
    print("saved", ART / f"{name}.pdf")

## Fig. 1 - study design

Counts, epochs and the channel correspondence come from this run. Also saved as SVG with editable text: open it in
Inkscape, or in PowerPoint (Insert > Pictures, then right-click > Convert to Shape) to retouch it.

In [ ]:
flow = pd.read_csv(out / "tables" / "F0_participant_flow.csv", encoding="utf-8-sig")
s1 = pd.read_csv(out / "tables" / "S1_brainlat_channel_map_1020.csv")
subj_bl, subj_gr = load_table(out / "derived" / "brainlat_subjects.csv"), load_table(out / "derived" / "greek_subjects.csv")
df_bl, df_gr = load_table(out / "derived" / "brainlat_epochs.csv"), load_table(out / "derived" / "greek_epochs.csv")
bl, gr = flow[flow.cohort == "BrainLat"], flow[flow.cohort == "Greek"]
step = lambda d, start: d[d.step.str.startswith(start)].iloc[0]
bl_set, bl_an, gr_an = step(bl, "Resting-state EEG file"), step(bl, "Analysable EEG"), step(gr, "Analysable EEG")
exc = [f"{int(bl_set[g] - bl_an[g])} {g}" for g in ("AD", "HC") if bl_set[g] - bl_an[g] > 0]
reason = re.sub(r"\s*\(n = \d+\)", "", str(bl_set.reason)) if pd.notna(bl_set.reason) else ""
info = {"n_bl_AD": int(bl_an.AD), "n_bl_HC": int(bl_an.HC),
        "bl_excluded_text": (" and ".join(exc) + f" excluded: {reason}") if exc else "no participant excluded",
        "n_gr_AD": int(gr_an.AD), "n_gr_HC": int(gr_an.HC), "n_gr_FTD": int(gr.FTD.dropna().iloc[0]),
        "epochs_bl": len(df_bl), "epochs_gr": len(df_gr),
        "sfreq_bl": float(subj_bl.sfreq.median()), "sfreq_gr": float(subj_gr.sfreq.median()),
        "angle_mean": float(s1.angle_deg.mean()), "angle_max": float(s1.angle_deg.max()),
        "rf_trees": ctx.rt["rf_trees"], "channels": ctx.channels, "brainlat_selected": list(s1.channel_brainlat)}
print(info | {"channels": "19", "brainlat_selected": "19"})
fig = study_design_figure(info)
fig.savefig(ART / "Fig1_study_design.svg")
save(fig, "Fig1_study_design")

## Figs. 2 and 3 - participant flow and ROC curves (same content as fig0 and fig1, article lettering)

In [ ]:
article_style()
save(fig_flow(flow), "Fig2_participant_flow")
EEG_LABEL, COMB_LABEL = "EEG score (RF, LOSO)", "Demographics + EEG score"
panels = []
for name, demo_label, other, title in (("BrainLat", "Age + sex + education", "RQ3b", "BrainLat"),
                                       ("Greek", "Age + sex", "RQ3a", "Greek cohort (ds004504)")):
    b = load_table(out / "predictions" / f"baselines_{name}.csv")
    get = lambda m: (b[b.model == m]["y"].to_numpy(), b[b.model == m]["p"].to_numpy())
    tp = load_table(out / "predictions" / f"transport_{other}.csv")
    panels.append({"title": title, "eeg": get(EEG_LABEL), "other": (tp["y"].to_numpy(), tp["p"].to_numpy()),
                   "demo": (demo_label, *get(demo_label)), "combined": get(COMB_LABEL)})
save(roc_figure(panels), "Fig3_roc_curves")

## Fig. 4 - RQ1, incremental value

(a) Odds ratio of the EEG score beyond the demographics in every analysis of RQ1. (b, c) Why the null bootstrap was
needed: the distribution of the statistic when the EEG adds nothing, obtained by re-running the whole
leave-one-subject-out procedure, against the observed value and the reference distribution of the naive test.

In [ ]:
article_style()
boot_bl = read_json(out / "tables" / "null_bootstrap_BrainLat.json")
boot_gr = read_json(out / "tables" / "null_bootstrap_Greek.json")
save(fig_rq1(inc, tr, boot_bl, boot_gr), "Fig4_RQ1_incremental_value")

## Fig. 5 - RQ2, confounding by design

(a) Discrimination when AD and controls are made comparable. (b, c) Whether the adjustment really made them
comparable: standardised differences in the demographics before and after matching or weighting.

In [ ]:
t2b = pd.read_csv(out / "tables" / "T2b_confounding_by_design.csv", encoding="utf-8-sig")
bal = {k: pd.read_csv(out / "tables" / f"S3_balance_{k}.csv", encoding="utf-8-sig")
       for k in ("BrainLat_matching", "BrainLat_overlap", "Greek_overlap")}
save(fig_rq2(t2b, bal["BrainLat_matching"], bal["BrainLat_overlap"], bal["Greek_overlap"]),
     "Fig5_RQ2_confounding_by_design")

## Fig. 6 - RQ3, transportability and calibration (with the post hoc recalibration)

In [ ]:
subj_a = load_table(out / "post_hoc" / "recalibrated_predictions_RQ3a.csv")
subj_b = load_table(out / "post_hoc" / "recalibrated_predictions_RQ3b.csv")
R = rec["results"]
save(fig_rq3(subj_a, R["RQ3a"], subj_b, R["RQ3b"], ctx.threshold, tr["TRANSP"]["RQ3a"]["AUC"],
             tr["TRANSP"]["RQ3b"]["AUC"]), "Fig6_RQ3_transportability")

## Fig. 7 - each feature in both cohorts (descriptive)

In [ ]:
rows = feature_rows(keys, ctx.features)
display(rows.round(3))
save(fig_features(rows, ctx.spectral), "Fig7_feature_consistency")

## Supplementary figures

In [ ]:
DESC = out / "descriptive"
article_style()
if (DESC / "montage_targets.csv").exists():
    save(fig_montage_error(pd.read_csv(DESC / "montage_electrodes.csv", keep_default_na=False),
                           pd.read_csv(DESC / "montage_targets.csv"), read_json(DESC / "montage_spacing.json")),
         "FigS1_montage_error")
else:
    print("WARNING: Fig. S1 skipped: run notebook 11.")
article_style()
if (DESC / "spectra_by_group.csv").exists():
    save(fig_spectra(pd.read_csv(DESC / "spectra_by_group.csv"),
                     plan["preprocessing"]["greek_band_harmonization"]["h_freq"]), "FigS2_group_spectra")
else:
    print("WARNING: Fig. S2 skipped: run notebook 11.")
article_style()
g = inc["INC"]["Greek"]
e_pt, e_ci = inc["E_value"]
edu = keys.get("rq1b.reference.education_or", {}).get("value")
save(fig_evalue(g["OR"], g["OR_lo"], e_pt, e_ci, edu), "FigS3_e_value")
ds = domain_shift_rows(df_bl, df_gr, ctx.features)
display(ds.round(2))
save(fig_domain_shift(ds), "FigS4_domain_shift")
article_style()
f_hp = out / "post_hoc" / "hyperparameters.json"
if f_hp.exists():
    hp = read_json(f_hp)
    if hp["mode"] != ctx.tag:
        raise RuntimeError(f"Notebook 10 ran in {hp['mode']} mode and this run is {ctx.tag}: run notebook 10 again.")
    save(fig_hyperparameters(hp["results"]), "FigS5_hyperparameters")
else:
    print("WARNING: Fig. S5 skipped: run notebook 10.")

## All tables of the article in one Word file, and the supplementary files

In [ ]:
f = write_article_tables(out, {"plan_hash": ctx.plan_hash, "mode": ctx.tag, "plan": plan})
print("Tables and figure legends:", f)
SUPP = ART / "supplementary_files"
SUPP.mkdir(exist_ok=True)
for name in ("analysis_plan.json", "post_hoc_plan.json", "post_hoc_plan_2.json"):
    shutil.copy(ctx.repo / "config" / name, SUPP / name)
print("\nEverything for the manuscript is in", ART)
for p in sorted(ART.rglob("*")):
    if p.is_file():
        print("  ", p.relative_to(ART))